# 1. [calculate] labour productivity
- Save to `working_yearly` new table
**Assets**
- Total assets: book value of all assets
(i.e. intangible and tangible assets, stock, current and non-currents assets)#
- Total liabilities: sum of current liabilities (i.e. loans and short-term debt, creditors and non-current liabilities (i.e. long-term financial liabilities including borrowing from credit institutions and bonds issued).
- Leverage: ratio of total liabilities to total assets.
  
**Income**
- Operating revenue (turnover): sum of net sales, other operating revenues and stock variations.
- Wage bill: renumeration_employees
- Employment: number of employees on the company’s payroll. 
- Negative turnover values. Turnover is defined as the operating revenue in FAME. In a few cases, some companies report negative turnover values. We flag (but keep) those companies reporting negative turnover values.  
   
**Productivity** 
- GVA (Lars): wage bill + EBITDA
- GVA (bottom-up): profit_loss_pretax + interest_paid + depreciation + remuneration_employees
- Productivity: GVA / employees
- Average wage: wage bill / employees
- Use lns

In [1]:
import ibis
from utils.f_0_dirs import get_data_dirs

old_table_name = "fame_yearly_kp"
new_table_name = "working_yearly"

# Initialize connection
dirs = get_data_dirs()
con = ibis.duckdb.connect(str(dirs.db_path))

# Reference the existing deflated table
fame_yearly = con.table(old_table_name)

# Calculate the new metrics using Ibis lazy evaluation
# We use ibis.ifelse to safely handle natural logarithms of negative or zero GVA
working_yearly = fame_yearly.mutate(
    gva1 = fame_yearly.wages + fame_yearly.ebitda,
    gva2 =  fame_yearly.profit_loss_pretax +
            fame_yearly.interest_paid +
            fame_yearly.depreciation +
            fame_yearly.remuneration_employees
).mutate(
    gva1_per_worker = ibis._.gva1 / fame_yearly.employees,
    gva2_per_worker = ibis._.gva2 / fame_yearly.employees,
    average_wage = fame_yearly.wages / fame_yearly.employees
)

# Verify the final materialized table
table_t_working = ibis.memtable(working_yearly)
print(f"\nSample of {new_table_name}:")
display(table_t_working.sample(0.0001).execute())


Sample of working_yearly:


,registered_number,year,consolidated,turnover,shareholders_funds,profit_loss_pretax,employees,tangibles,tangibles_land_and_buildings,tangibles_land_freehold,...,social_security_costs,pensions_costs,other_staff_costs,renumeration_directors,ebitda,gva1,gva2,gva1_per_worker,gva2_per_worker,average_wage
0,NI032950,2006,False,320.590624,40.109499,13.281542,11,15.471483,NaN,NaN,...,9.392797,NaN,NaN,NaN,18.720920,168.073417,175.110837,15.279402,15.919167,13.577500
1,02319033,2006,False,11290.223340,2758.699510,1270.526724,13,34.966528,NaN,NaN,...,62.822189,35.598914,NaN,58.087,1231.196080,1843.420992,NaN,141.801615,NaN,47.094224
2,00293529,2006,False,89439.138088,97197.740516,37501.751138,513,19696.854325,8889.047041,NaN,...,2839.227273,2465.343980,NaN,366.000,12460.116844,39991.968195,70470.173743,77.957053,137.368760,53.668326
3,01417048,2006,False,443664.036419,378706.828528,41780.728376,2395,176582.701062,47298.937785,NaN,...,9142.997543,8326.658477,NaN,983.000,52659.484067,160416.240824,180707.141153,66.979641,75.451833,44.992383
4,03425239,2006,False,NaN,1365.693763,101.505097,115,338.542147,115.330577,NaN,...,611.957152,62.732392,NaN,NaN,299.118483,6587.864849,NaN,57.285781,NaN,54.684751
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
107,04420052,2024,False,27679.000000,11966.000000,-1043.000000,416,1203.000000,NaN,NaN,...,1620.000000,2306.000000,12.0,282.000,-810.000000,14701.000000,NaN,35.338942,NaN,37.286058
108,03175969,2024,False,1345.693000,291.233000,14.242000,58,26.642000,NaN,NaN,...,69.639000,20.172000,NaN,NaN,22.975000,1045.833000,NaN,18.031603,NaN,17.635483
109,05108932,2024,False,16313.446000,4144.468000,2469.990000,112,172.869000,62.303000,NaN,...,251.495000,49.120000,NaN,NaN,2587.191000,5373.382000,NaN,47.976625,NaN,24.876705
110,00471436,2024,False,3241.066000,13359.540000,82.606000,54,9217.970000,NaN,NaN,...,35.308000,33.657000,NaN,NaN,181.447000,845.449000,NaN,15.656463,NaN,12.296333


In [2]:
working_yearly_skinny = table_t_working.select(
    "registered_number", "year",
    "employees", "fixed_total", "total_assets",
    "average_wage", "gva1", "gva2",
    "gva1_per_worker", "gva2_per_worker"
)

print(f"✅ Inserting columns into new '{new_table_name}' table: {working_yearly_skinny.columns}")
con.create_table(new_table_name, working_yearly_skinny, overwrite=True)

# Verify the final materialized table
final_table = con.table(new_table_name)
row_count = final_table.count().execute()
col_count = len(final_table.columns)

print(f"✅ Materialized '{new_table_name}' table.")
print(f"📊 Number of rows: {row_count:,}")
print(f"📊 Number of columns: {col_count}")
print(f"\nHead of {new_table_name}:")
display(final_table.sample(200 / row_count).execute())

✅ Inserting columns into new 'working_yearly' table: ('registered_number', 'year', 'employees', 'fixed_total', 'total_assets', 'average_wage', 'gva1', 'gva2', 'gva1_per_worker', 'gva2_per_worker')
✅ Materialized 'working_yearly' table.
📊 Number of rows: 1,128,490
📊 Number of columns: 10

Head of working_yearly:


,registered_number,year,employees,fixed_total,total_assets,average_wage,gva1,gva2,gva1_per_worker,gva2_per_worker
0,01911662,2006,14,70.976786,222.628217,21.701441,-212.988515,357.460490,-15.213465,25.532892
1,02457325,2006,42,93.746495,1113.814307,74.139836,3276.638228,NaN,78.015196,NaN
2,02132476,2006,60,4811.832882,13672.551012,19.005598,1624.051079,1770.759887,27.067518,29.512665
3,00613551,2006,209,59530.443096,250939.784522,48.331960,34897.059425,29526.378308,166.971576,141.274537
4,02958269,2006,47,5476.234754,5877.826385,29.384628,1487.142249,1323.206545,31.641324,28.153331
...,...,...,...,...,...,...,...,...,...,...
188,04853546,2024,79,8413.000000,113695.000000,35.708861,-131.000000,-1086.000000,-1.658228,-13.746835
189,10319988,2024,66,81.271000,9624.046000,58.941894,-3584.703000,NaN,-54.313682,NaN
190,07129804,2024,178,422.963000,2913.493000,24.943135,4477.793000,4844.943000,25.156140,27.218781
191,SC179833,2024,83,2052.176000,4802.208000,31.815602,3260.205000,3429.820000,39.279578,41.323133


# [calculate] 2. TFP

$$\ln(Y_{it}) = \alpha_i + \gamma_t + \beta_K \ln(K_{it}) + \beta_L \ln(L_{it}) + \varepsilon_{it}$$
- $Y_{it}$: `gva1` or `gva2`
- $K_{it}$: `fixed_total` or `total_assets`
- $L_{it}$: `employees`  
### Capital choice
- `fixed_total` = `tangibles` + `intangibles` + `investments_other`, representing different types of capitals
- `total_assets` = `fixed_total` + `current_assets`, which includes non-productive current_assets (e.g. cash, stock, debtors) and productive current_assets (e.g. stock of raw materials, work in progress and finished goods).

In [3]:
import ibis
from ibis import _
import pandas as pd
import numpy as np
import statsmodels.api as sm
from linearmodels.panel import PanelOLS
from utils.f_0_dirs import get_data_dirs

dirs = get_data_dirs(segment="descriptives")
con = ibis.duckdb.connect(str(dirs.db_path))
table_working = con.table("working_yearly")
table_results = table_working.select("registered_number", "year")

# 3. Define the 4 model setups to iterate through
# Varying Y (gva1 vs gva2) and K (fixed_total vs tangibles)
models = {
    'tfp1': {'Y': 'gva1', 'K': 'total_assets', 'L': 'employees'},
    'tfp2': {'Y': 'gva2', 'K': 'total_assets', 'L': 'employees'},
    'tfp3': {'Y': 'gva1', 'K': 'fixed_total', 'L': 'employees'}
}

# Dictionary to store the parameter outputs (\beta_K, \beta_L) and model summaries
parameter_tables = {}

for name, mod in models.items():

    # Mutate to dynamically log-transform all columns, adding ln_ prefix to column name
    table_skinny = (
        table_working
        .select(["registered_number", "year"] + list(mod.values()))
        .rename(mod)
    )
    table_start = table_skinny
    for col in ['Y', 'K', 'L']:
        table_filtered = table_start.filter(_[col] > 0)
        table_logged = table_filtered.mutate(**{f'ln_{col}': np.log(_[col]) }) # type: ignore
        table_start = table_logged

    # 1. Execute into a Pandas DataFrame and set the MultiIndex for linearmodels
    df_model = table_start.execute().set_index(['registered_number', 'year'])

    # Define Endogenous (Y) and Exogenous (X) variables
    Y = df_model['ln_Y']
    X = sm.add_constant(df_model[['ln_K', 'ln_L']])
    
    # 2. Estimate the model with Firm and Year Fixed Effects
    mod_ols = PanelOLS(Y, X, entity_effects=True, time_effects=True)
    
    # Fit model with firm-clustered standard errors
    res = mod_ols.fit(cov_type='clustered', cluster_entity=True)
    
    # Extract \beta_K and \beta_L
    beta_K = res.params['ln_K']
    beta_L = res.params['ln_L']
    
    # 3. Calculate firm-year specific TFP (Solow Residual) inside the Ibis pipeline
    # TFP_it = ln(Y_it) - \beta_K*ln(K_it) - \beta_L*ln(L_it)
    table_with_tfp = table_start.mutate(
        **{name: _['ln_Y'] - (beta_K * _['ln_K']) - (beta_L * _['ln_L'])}
    )
    
    # 4. Join the calculated TFP column back to the main dataframe
    # We select only the keys and the new TFP column to avoid duplicating ln_ columns
    table_results = (
        table_results
        .left_join(
            table_with_tfp,
            ["registered_number", "year"],
            rname='{name}_' + name
        )
        .drop("registered_number_" + name, "year_" + name) # Drop duplicate join keys
    )
    
    # 5. Store the results and parameters
    parameter_tables[name] = {
        'beta_K': beta_K,
        'beta_L': beta_L,
        # Safely extract time effects if they exist
        'gamma_t': res.estimated_effects.xs('time_effects', level=1) if 'time_effects' in res.estimated_effects.index.names else None, 
        'summary': res.summary
    }
    print(f"✅ Model '{name}' estimated: beta_K={beta_K:.4f}, beta_L={beta_L:.4f}")

print(f"Panel regressions complete. {len(models)} TFP variants added to the dataframe.")

✅ Model 'tfp1' estimated: beta_K=0.2864, beta_L=0.6169
✅ Model 'tfp2' estimated: beta_K=0.2654, beta_L=0.6109
✅ Model 'tfp3' estimated: beta_K=0.0651, beta_L=0.7197
Panel regressions complete. 3 TFP variants added to the dataframe.


In [4]:
# From the above cell, display the revised table with the new TFP columns and the parameter estimates for each model
table_sample = table_results.sample(0.0001).execute()
display(table_sample)

# Send parameter_tables to a markdown file in dirs.output_dir to easily compare
output_file = dirs.output_dir / f"tfp_2factor_results_{name}.md"
with open(output_file, 'w') as f:
    for name, params in parameter_tables.items():
        # Write all to 1 big markdown file
        # Just write the default display(params) output to the file
        f.write(f"# Parameters for model '{name}'\n\n")
        f.write(f"## Estimated Coefficients\n")
        f.write(f"- beta_K: {params['beta_K']:.6f}\n")
        f.write(f"- beta_L: {params['beta_L']:.6f}\n")
        if params['gamma_t'] is not None:
            f.write(f"\n## Time Effects (gamma_t)\n")
            f.write(params['gamma_t'].to_markdown())
        f.write("\n\n## Model Summary\n")
        f.write(params['summary'].as_text())
        print(f"✅ Parameters for model '{name}' written to {output_file}")

# Verify that that \ln Y = \alpha_i + \gamma_t + \beta_K \ln K + \beta_L \ln L + TFP_it holds for this sample
name, mod = list(models.items())[0]
params = parameter_tables[name]
for row in table_sample.itertuples():
    # Get TFP, L, K
    tfp = row._asdict()[name]
    ln_K = row.ln_K
    ln_L = row.ln_L
    ln_Y = row.ln_Y
    if any(np.isnan([tfp, ln_K, ln_L, ln_Y])):
        print(f"Skipping row {row.Index} due to NaN values.")
        continue
    ln_Y_calc = tfp + params['beta_K'] * ln_K + params['beta_L'] * ln_L
    diff = ln_Y - ln_Y_calc

    assert np.isclose(diff, 0, atol=1e-6), (
        f"TFP calculation check failed for row {row.Index}!  \
        Expected ln_Y: {ln_Y:.6f}, Calculated ln_Y: {ln_Y_calc:.6f}, Difference: {diff:.6e}"
    )

# Count number of tfp1 and tfp2 observations in the results table (as a percentage of total rows)
# Output as a pd dataframe
total_rows = table_results.count().execute()
tfp1_count = table_results.filter(_['tfp1'].isnull() == False).count().execute()
tfp2_count = table_results.filter(_['tfp2'].isnull() == False).count().execute()
tfp3_count = table_results.filter(_['tfp3'].isnull() == False).count().execute()
tfp_counts = pd.DataFrame({
    'TFP Variant': ['tfp1', 'tfp2', 'tfp3'],
    'Count': [tfp1_count, tfp2_count, tfp3_count]
})
tfp_counts['Percentage'] = tfp_counts['Count'] / total_rows * 100
print(f"\nTFP Counts and Percentages (out of {total_rows:,} total rows):")
display(tfp_counts)

,registered_number,year,Y,K,L,ln_Y,ln_K,ln_L,tfp1,Y_tfp2,...,ln_K_tfp2,ln_L_tfp2,tfp2,Y_tfp3,K_tfp3,L_tfp3,ln_Y_tfp3,ln_K_tfp3,ln_L_tfp3,tfp3
0,SC208842,2006,4257.837598,24715.205405,91.0,8.356517,10.115174,4.510860,2.676981,3334.698209,...,10.115174,4.510860,2.671378,4257.837598,438.003930,91.0,8.356517,6.082228,4.510860,4.714122
1,00833384,2006,7067.858715,9565.092335,103.0,8.863313,9.165876,4.634729,3.379227,7172.644225,...,9.165876,4.634729,3.613585,7067.858715,2624.693736,103.0,8.863313,7.872719,4.634729,5.015230
2,03276253,2006,1806.179256,3979.508744,53.0,7.498969,8.288914,3.970292,2.675916,2461.749864,...,8.288914,3.970292,3.182880,1806.179256,887.074235,53.0,7.498969,6.787929,3.970292,4.199696
3,00785779,2006,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15640.792963,...,10.267375,4.948760,3.908960,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,02943591,2006,2787.010320,9374.570637,45.0,7.932725,9.145756,3.806662,2.965228,2965.114662,...,9.145756,3.806662,3.241440,2787.010320,1584.326797,45.0,7.932725,7.367915,3.806662,4.713468
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119,00521505,2007,1042.296482,10179.981511,51.0,6.949182,9.228178,3.931826,1.880868,NaN,...,NaN,NaN,NaN,1042.296482,10009.878840,51.0,6.949182,9.211328,3.931826,3.519861
120,05060103,2014,1574.952356,2925.490610,22.0,7.361980,7.981217,3.091042,3.169449,NaN,...,NaN,NaN,NaN,1574.952356,987.858706,22.0,7.361980,6.895540,3.091042,4.688509
121,00897432,2015,4364.854717,9864.353803,78.0,8.381340,9.196683,4.356709,3.059940,NaN,...,NaN,NaN,NaN,4364.854717,6268.530453,78.0,8.381340,8.743297,4.356709,4.676689
122,07070078,2016,343.751963,1160.539446,16.0,5.839920,7.056640,2.772589,2.108624,NaN,...,NaN,NaN,NaN,343.751963,471.376242,16.0,5.839920,6.155657,2.772589,3.443801


✅ Parameters for model 'tfp1' written to /mnt/c/Users/lazym/Documents/Code/dissertation/descriptives/output/tfp_2factor_results_tfp3.md
✅ Parameters for model 'tfp2' written to /mnt/c/Users/lazym/Documents/Code/dissertation/descriptives/output/tfp_2factor_results_tfp3.md
✅ Parameters for model 'tfp3' written to /mnt/c/Users/lazym/Documents/Code/dissertation/descriptives/output/tfp_2factor_results_tfp3.md
Skipping row 3 due to NaN values.
Skipping row 68 due to NaN values.
Skipping row 84 due to NaN values.
Skipping row 93 due to NaN values.
Skipping row 94 due to NaN values.
Skipping row 96 due to NaN values.
Skipping row 97 due to NaN values.
Skipping row 107 due to NaN values.
Skipping row 108 due to NaN values.
Skipping row 109 due to NaN values.
Skipping row 113 due to NaN values.
Skipping row 114 due to NaN values.
Skipping row 117 due to NaN values.

TFP Counts and Percentages (out of 1,128,490 total rows):


,TFP Variant,Count,Percentage
0,tfp1,1045164,92.616151
1,tfp2,603586,53.486163
2,tfp3,1006904,89.225780


In [5]:
preferred_model = 'tfp1'
# Add the tfp1 column from this table_results to the working_yearly table in the database
# Rename as 'tfp'
table_with_tfp = (
    table_working
    .left_join(
        table_results.select(['registered_number', 'year', preferred_model]),
        ['registered_number', 'year']
    )
    # Rename the preferred TFP column to 'tfp' for clarity
    .rename(tfp=preferred_model)
    .drop("registered_number_right", "year_right")
)
display(table_with_tfp.sample(0.0001).execute())

,registered_number,year,employees,fixed_total,total_assets,average_wage,gva1,gva2,gva1_per_worker,gva2_per_worker,tfp
0,03957086,2006,258,3426.019727,44257.616085,68.243415,20783.712971,25965.830789,80.557027,100.642755,3.452676
1,02805730,2006,43,4760.253390,12320.633853,34.985521,2028.271458,2091.720297,47.169104,48.644658,2.597226
2,02295585,2006,28,158.752578,5264.362316,45.861404,1431.903265,1696.130843,51.139402,60.576102,2.757207
3,01413878,2006,1216,70982.617998,135359.564097,26.243323,2944.562637,NaN,2.421515,NaN,0.221910
4,03603234,2006,58,203161.550835,636607.440061,92.781157,-6848.621555,8416.501691,-118.079682,145.112098,NaN
...,...,...,...,...,...,...,...,...,...,...,...
118,08423282,2023,43,1459.166405,16663.550574,41.519077,1508.585781,-2175.451613,35.083390,-50.591898,2.214742
119,01053323,2024,116,893.920000,6349.488000,39.601724,5822.952000,NaN,50.197862,NaN,3.229497
120,13438798,2024,269,93186.849000,97181.686000,29.359758,15157.477000,10544.129000,56.347498,39.197506,2.885987
121,06431269,2024,13,4.220000,543.870000,20.762154,370.110000,NaN,28.470000,NaN,2.527654


In [ ]:
overwrite = True
if overwrite:
    # Safe overwrite of the working_yearly table with the new tfp column
    con.create_table("working_yearly_temp", table_with_tfp)
    con.create_table("working_yearly", con.table("working_yearly_temp"), overwrite=True)
    con.drop_table("working_yearly_temp")

    # Verify schema and rows of the final working_yearly table
    final_table = con.table("working_yearly")
    row_count = final_table.count().execute()
    col_count = len(final_table.columns)
    tfp_col_exists = 'tfp' in final_table.columns
    tfp_nan_count = final_table.filter(_['tfp'].isnull() == True).count().execute() if tfp_col_exists else None
    print(f"\nFinal 'working_yearly' table verification:"
        f"\n- Rows: {row_count}"
        f"\n- Columns: {col_count}"
        f"\n- TFP column exists: {tfp_col_exists}"
        f"\n- TFP NaN count: {tfp_nan_count}")